In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import cv2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet import preprocess_input
import gradio as gr

In [ ]:
import tensorflow as tf
import os
import numpy as np
from collections import Counter

data_dir = '/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented'
img_size = (224, 224)
batch_size = 32

class_names = sorted(os.listdir(data_dir))
num_classes = len(class_names)

file_paths = []
labels = []

for idx, class_name in enumerate(class_names):
    class_path = os.path.join(data_dir, class_name)
    if os.path.isdir(class_path):
        files = [os.path.join(class_path, f) for f in os.listdir(class_path)]
        file_paths.extend(files)
        labels.extend([idx] * len(files))

file_paths = np.array(file_paths)
labels = np.array(labels)

class_counts = Counter(labels)
target_count = int(np.mean(list(class_counts.values())))

balanced_paths = []
balanced_labels = []

for class_idx in range(num_classes):
    class_indices = np.where(labels == class_idx)[0]
    class_files = file_paths[class_indices]
    
    if len(class_files) < target_count:
        extra = np.random.choice(class_files, target_count - len(class_files), replace=True)
        new_files = np.concatenate([class_files, extra])
    else:
        new_files = np.random.choice(class_files, target_count, replace=False)
    
    balanced_paths.extend(new_files)
    balanced_labels.extend([class_idx] * target_count)

def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, img_size)
    img = img / 255.0
    
    label = tf.one_hot(label, num_classes)
    return img, label

full_dataset = tf.data.Dataset.from_tensor_slices((balanced_paths, balanced_labels))
full_dataset = full_dataset.shuffle(len(balanced_paths), reshuffle_each_iteration=False)

val_size = int(len(balanced_paths) * 0.2)
train_ds = full_dataset.skip(val_size)
val_ds = full_dataset.take(val_size)

def configure_for_performance(ds):
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_data = configure_for_performance(train_ds)
val_data = configure_for_performance(val_ds)

print(f"Original distribution: {class_counts}")
print(f"Balanced samples per class: {target_count}")
for x, y in train_data.take(1):
    print(f"Image batch shape: {x.shape}")
    print(f"Label batch shape (Categorical): {y.shape}")


In [ ]:
pretrained_model = ResNet50(weights='imagenet', include_top=False, input_shape = (224, 224, 3))

In [ ]:
x = pretrained_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
outputs = Dense(6, activation='softmax')(x)

In [ ]:
model = Model(inputs=pretrained_model.input, outputs=outputs)

In [ ]:
for layer in pretrained_model.layers[:-50]:
    layer.trainable = False

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    class_weight=class_weights
)

In [ ]:
preds = model.predict(val_data)
y_pred = np.argmax(preds, axis=1)

In [ ]:
model.save('model2.keras')

In [ ]:
train_acc = history.history['accuracy'][0]
val_acc = history.history['val_accuracy'][0]
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
import matplotlib.pyplot as plt
labels = ['Training', 'Validation']
values = [train_acc, val_acc]
plt.bar(labels, values, color=['blue', 'red'])
plt.ylabel('Accuracy')
plt.title('Accuracy After 1 Epoch')
plt.ylim(0, 1)
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
plt.show()

In [ ]:
from tensorflow.keras.utils import load_img,img_to_array
img = load_img("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented/Non_Damage/10004.jpg", target_size=(224,224))
print(img)
img = img_to_array(img)/255.0
img = np.expand_dims(img, axis=0)

In [ ]:
def predict(img):
    img = load_img(img, target_size=(224,224))
    img = img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)
    pred_class = int(np.argmax(pred))
    return className(pred_class)

In [ ]:
pred = model.predict(img)
pred_class = int(np.argmax(pred))
confidence = float(np.max(pred))

In [ ]:
def className(n):
    if n == 0:
        return "Damaged_Infrastrucutre"
    elif n == 1:
        return "Fire_Disaster"
    elif n == 2:
        return "Human_Damage"
    elif n == 3:
        return "Land_Disaster"
    elif n == 4:
        return "Non_Damage"
    else :
        return "Water_Disaster"

In [ ]:
df = pd.DataFrame([{
    "prediction": className(pred_class),
}])

In [ ]:
df

In [ ]:
import gradio as gr

In [ ]:
demo = gr.Interface(
    fn=predict, 
    inputs=gr.Image(type="filepath"), 
    outputs=gr.Label()
)
demo.launch()